In [1]:
import yfinance as yf
import os
import json
from pathlib import Path

In [2]:
TICKERS_TO_DWNLD = ["NVDA", "^SPX", "BTC-USD", "^IXIC", "CIAFF", "TSLA", "TSM",
"PYPL", "VZ", "IBM", "2222.SR", "ETH-USD", "LLOY.L", "SHEL.L", "UKOG.L", 
"CHD", "TSN", "INTC", "AMD", "AAPL", "AMZN", "JPM", "ORCL", "PG", "XOM",
"MSFT", "META", "NFLX", "WMT", "LLOY", "HSBA", "BA.", "0700.HK", "1810.HK"
]

INTERVAL = "4h"
PERIOD = "2y"
IND_WINDOW = 14 # Reproduce all the .csv's in case you change it 

config = {
    "INTERVAL": INTERVAL,
    "PERIOD": PERIOD,
    "IND_WINDOW": IND_WINDOW
}

with open("global_vars.json", "w") as f:
    json.dump(config, f, indent=4)

path = Path(f"stocks/{PERIOD}/{INTERVAL}")
if not path.exists(): path.mkdir(parents=True, exist_ok=True)

In [3]:
def calculate_rsi(series):
    delta = series.diff()

    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.ewm(alpha=1/IND_WINDOW, min_periods=IND_WINDOW).mean()
    avg_loss = loss.ewm(alpha=1/IND_WINDOW, min_periods=IND_WINDOW).mean()

    return 100 - (100 / (1 + (avg_gain/avg_loss)))

def stochastic_oscillator(df, smoothing=3):
    low_min = df["Low"].rolling(window=IND_WINDOW).min()
    high_max = df["High"].rolling(window=IND_WINDOW).max()

    df['%K'] = ((df["Close"] - low_min) / (high_max - low_min)) * 100
    df['%D'] = df['%K'].rolling(window=smoothing).mean()
    return df

In [4]:
for name in TICKERS_TO_DWNLD:
    try:
        ticker = yf.Ticker(name)
        dividends = ticker.dividends

        price_data = ticker.history(period=PERIOD, interval=INTERVAL)

    except:
        print("Could not install " + name)
        continue

    # price_data["Price"] = (price_data["High"] + price_data["Low"])/2
    price_data["Dividends"] = (price_data["Dividends"] != 0.0).astype(float)
    price_data["Rsi"] = calculate_rsi(price_data["Close"])
    stochastic_oscillator(price_data)

    price_data = price_data.drop(columns=["High", "Low", "Open", "Volume", "Stock Splits"])
    price_data = price_data.iloc[IND_WINDOW + 1:]
    price_data.to_csv(f"stocks/{PERIOD}/{INTERVAL}/{name}.csv")

$^SPX: possibly delisted; no price data found  (1d 1926-11-08 -> 2025-10-14)
$IBM: possibly delisted; no price data found  (1d 1926-11-08 -> 2025-10-14)
HTTP Error 404: 
$LLOY: possibly delisted; no timezone found
$LLOY: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")


KeyError: 'Dividends'